---

![Portada](img/usacc.jpg)

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">


<h1 align="center"><b> 🚗 Muertes Accidentales Mensuales en Estados Unidos </b></h1>

---

El conjunto de datos **USAccDeaths**, de la librería **datasets** de R, registra el total mensual de muertes accidentales en Estados Unidos entre enero de 1973 y diciembre de 1978 (72 observaciones), tomado de Brockwell y Davis (1991). La serie se caracteriza por una estacionalidad anual muy marcada ($S=12$), con máximos en los meses de verano (julio) y mínimos en invierno (febrero), por lo que es un ejemplo adecuado para identificar, estimar, validar y pronosticar un modelo **SARIMA** $(p,d,q) \times (P,D,Q)_{12}$ siguiendo la metodología de **Box-Jenkins**.

**Fases de modelación:**
1. Análisis descriptivo
2. Transformación de Box-Cox
3. Prueba de estacionariedad
4. Identificación (ACF y PACF)
5. Identificación con criterios de información
6. Estimación de dos modelos tentativos
7. Diagnóstico residual y normalidad
8. Selección final del modelo
9. Pronóstico
10. Evaluación del pronóstico (Train / Test)
11. Aplicación Shiny interactiva

</div>

---

## **Librerías y Carga de los Datos**

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">


<h1 align="center"><b> </b></h1>

---

</div>

In [ ]:
# Instalar Librerías (solo la primera vez; si ya están instaladas, NO correr esta celda) #
install.packages("stats")
install.packages("astsa")
install.packages("forecast")
install.packages("tseries")
install.packages("nortest")
install.packages("fBasics")
install.packages("car")
install.packages("MLmetrics")
install.packages("matrixcalc")
install.packages("aTSA")
install.packages("MASS")
install.packages("TSA")
install.packages("pastecs")
install.packages("ggplot2")

In [ ]:
# Cargar Librerías #
library(stats)
library(astsa)
library(forecast)
library(tseries)
library(nortest)
library(fBasics)
library(car)
library(MLmetrics)
library(matrixcalc)
library(aTSA)
library(MASS)
library(TSA)
library(pastecs)
library(ggplot2)

# Nota: aTSA y TSA enmascaran funciones con el mismo nombre (adf.test, arima),
# por eso más adelante se usa tseries::adf.test() y stats::arima() de forma explícita.

In [ ]:
# Cargar la serie USAccDeaths (librería datasets de R) como objeto ts() #
data("USAccDeaths")
accid <- USAccDeaths
print(accid)

## **Análisis Descriptivo**

In [ ]:
# Graficar la serie #
options(repr.plot.width=14, repr.plot.height=8)
ts.plot(accid, ylab="Número de muertes", xlab="Año",
        main="Serie de Muertes Accidentales Mensuales en EE. UU. (1973-1978)", lwd=2); grid(col = "black")

In [ ]:
# Medidas descriptivas de la serie (pastecs) #
round(stat.desc(as.numeric(accid)), 2)

In [ ]:
# Tabla Buys-Ballot library(pastecs) #
tb_bb = buysbal(accid, y=NULL, frequency=12, dateformat="m/Y")
tb_bb

In [ ]:
# Graficar la serie con periodos de cada 12 meses #
ts.plot(accid, ylab="Número de muertes", xlab="Año",
        main="Serie de Muertes Accidentales con Periodos Anuales", lwd=2); grid(col = "black")
abline(v=c(seq(1973,1979)), col="red")

In [ ]:
# Graficos estacionales #
par(mfrow=c(3,1))
ts.plot(accid,  main="Muertes accidentales mensuales")
monthplot(accid, main="Muertes accidentales para cada mes con el promedio")
seasonplot(accid, main="Perfil estacional", year.labels = TRUE, col = 1:6)

# Un perfil estacional es como poner todos los años uno encima del otro para ver qué partes del año se comportan igual y cuáles no #

In [ ]:
# Descomposición clásica de la serie (tendencia, estacionalidad y componente aleatorio) #

# 1. Descomposición aditiva
descomp <- decompose(accid)

# 2. Extraer tiempo numérico (Años)
tiempo <- as.numeric(time(accid))

# 3. Crear data frame unificado con los 4 componentes
df_descomp <- data.frame(
  Tiempo = rep(tiempo, 4),
  Valor = c(
    as.numeric(descomp$x),
    as.numeric(descomp$trend),
    as.numeric(descomp$seasonal),
    as.numeric(descomp$random)
  ),
  Componente = factor(
    rep(c("Serie Observada", "Tendencia", "Estacionalidad", "Residuos (Aleatorio)"), each = length(tiempo)),
    levels = c("Serie Observada", "Tendencia", "Estacionalidad", "Residuos (Aleatorio)")
  )
)

# 4. Gráfica en ggplot2
ggplot(df_descomp, aes(x = Tiempo, y = Valor)) +
  geom_hline(data = subset(df_descomp, Componente %in% c("Estacionalidad", "Residuos (Aleatorio)")),
             aes(yintercept = 0), color = "gray60", linetype = "dashed") +
  geom_line(color = "steelblue", linewidth = 0.7, na.rm = TRUE) +
  facet_wrap(~ Componente, ncol = 1, scales = "free_y", strip.position = "right") +
  scale_x_continuous(breaks = seq(1973, 1979, by = 1)) +
  labs(
    title = "Descomposición Clásica de la Serie de Muertes Accidentales",
    subtitle = "Descomposición aditiva con periodo S = 12",
    x = "Año",
    y = ""
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 10),
    strip.background = element_rect(fill = "gray95", color = NA),
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray92")
  )

---
## **Conclusión**:

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

La serie presenta un **patrón estacional anual muy marcado** ($S=12$): todos los años las muertes accidentales alcanzan su máximo en **julio** y su mínimo en **febrero**, como se aprecia en la tabla Buys-Ballot, el `monthplot` y el perfil estacional, donde las curvas de los seis años tienen la misma forma. La tendencia no es monótona: la serie **disminuye entre 1973 y 1976** y luego **vuelve a aumentar hacia 1978**, por lo que la media no es constante. La amplitud de la oscilación estacional se mantiene aproximadamente estable en el tiempo, lo cual se verificará formalmente con la transformación de Box-Cox.

</div>

---

## **Transformación de Box-Cox**

In [ ]:
# Alternativa 1 (car) #
accid_BC = powerTransform(accid)
summary(accid_BC)

In [ ]:
# Alternativa 2 (MASS) #
bc = boxcox(accid ~ 1, lambda = seq(-2, 2, 1/10), plotit = TRUE,
            xlab = expression(lambda), ylab = "Transformación Box-Cox")

# Lambda que maximiza la log-verosimilitud e intervalo de confianza del 95% #
lambda_bc = bc$x[which.max(bc$y)]
ic_lambda = range(bc$x[bc$y > max(bc$y) - qchisq(0.95, 1)/2])
cat("Lambda estimado:", lambda_bc, "\n")
cat("IC 95% para lambda: [", ic_lambda[1], ",", ic_lambda[2], "]\n")

In [ ]:
# Decisión sobre la transformación #
# - Si el IC contiene a 1  -> no se transforma (la varianza ya es estable)
# - Si el IC contiene a 0  -> transformación logarítmica
if (ic_lambda[1] <= 1 & ic_lambda[2] >= 1) {
  transf = "ninguna"
} else if (ic_lambda[1] <= 0 & ic_lambda[2] >= 0) {
  transf = "log"
} else {
  transf = ifelse(lambda_bc < 0.5, "log", "ninguna")
}

if (transf == "log") { accidt = log(accid) } else { accidt = accid }
cat("Transformación seleccionada:", transf, "\n")

In [ ]:
# Graficar serie original y serie con la que se va a modelar #
par(mfrow=c(2,1))
plot(accid, main="Serie Original", lwd=2); grid(col = "black")
plot(accidt, main=paste("Serie a modelar (transformación:", transf, ")"), lwd=2); grid(col = "black")

---
## **Conclusión**:

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

El valor estimado es $\hat{\lambda} =$ **___** con intervalo de confianza del 95% **[___ , ___]**. *(Completar con la salida.)* Si el intervalo contiene a $\lambda = 1$, **no es necesario transformar** la serie, pues la varianza es aproximadamente constante; si contiene a $\lambda = 0$, se usa la **transformación logarítmica**, como en el ejemplo de Air Passengers. De aquí en adelante se modela la serie `accidt`.

</div>

---

### **Prueba de Estacionaridad**

In [ ]:
# Aplicar la prueba de Dickey-Fuller de Estacionaridad #
# H0: La serie tiene raíz unitaria (no es estacionaria)
# H1: La serie es estacionaria
resultado_adf <- suppressWarnings(tseries::adf.test(accidt))

print(resultado_adf)

if (resultado_adf$p.value < 0.05) {
  print("La serie es estacionaria")
} else {
  print("La serie no es estacionaria")
}

In [ ]:
# Número de diferencias ordinarias (d) y estacionales (D) sugeridas (forecast) #
cat("Diferencias ordinarias sugeridas  (d):", ndiffs(accidt), "\n")
cat("Diferencias estacionales sugeridas (D):", nsdiffs(accidt), "\n")

## **Funciones de Autocorrelación y Autocorrelación Parcial**

In [ ]:
# ACF y PACF muestrales de la serie #
par(mfrow=c(2,1))
acf(as.numeric(accidt), lag.max=48, main="ACF de la serie")     # ACF hasta el rezago 48
pacf(as.numeric(accidt), lag.max=48, main="PACF de la serie")   # PACF hasta el rezago 48

In [ ]:
# ACF y PACF muestrales de la primera diferencia ordinaria, (1-B) #
par(mfrow=c(2,1))
acf(as.numeric(diff(accidt)), lag.max=48, main="ACF de la primera diferencia ordinaria")
pacf(as.numeric(diff(accidt)), lag.max=48, main="PACF de la primera diferencia ordinaria")

In [ ]:
# ACF y PACF muestrales de la primera diferencia estacional (1-B^12) #
par(mfrow=c(2,1))
acf(as.numeric(diff(accidt, lag=12)), lag.max=48, main = "ACF de la diferencia estacional 12")
pacf(as.numeric(diff(accidt, lag=12)), lag.max=48, main = "PACF de la diferencia estacional 12")

In [ ]:
# ACF y PACF muestrales de (1-B)(1-B^12) #
d1accidt = diff(accidt)
dd_accidt = diff(d1accidt, lag=12)
par(mfrow = c(2,1))
acf(as.numeric(dd_accidt), lag.max=48, main = "ACF de la diferencia ordinaria y estacional 12")
pacf(as.numeric(dd_accidt), lag.max=48, main = "PACF de la diferencia ordinaria y estacional 12")

In [ ]:
# Serie con diferencia ordinaria y estacional, (1-B)(1-B^12) #
ts.plot(dd_accidt, main="Serie diferenciada (1-B)(1-B^12)", ylab="", lwd=2); grid(col = "black")
abline(h=0, col="red", lty=2)

# Prueba de Dickey-Fuller sobre la serie diferenciada #
suppressWarnings(tseries::adf.test(dd_accidt))

---
## **Conclusión**:

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

En la ACF de la serie original aparecen **picos en los rezagos 12, 24 y 36 que decaen lentamente**, evidencia de una **estacionalidad no estacionaria**, por lo que se requiere una diferencia estacional ($D=1$). Al aplicar solo la diferencia estacional persiste un decaimiento lento asociado al nivel, que se corrige con una diferencia ordinaria ($d=1$). En la serie $(1-B)(1-B^{12})$ la ACF muestra un **corte en el rezago 1** y un **pico significativo en el rezago 12**, lo que sugiere una componente $MA(1)$ ordinaria y una $MA(1)$ estacional.

**Hay evidencia de un** $SARIMA(0,1,1)\times(0,1,1)_{12}$ *(verificar con los gráficos obtenidos)*.

</div>

---

## **Identificación con Criterios de Información**

In [ ]:
# Función que busca modelos según los criterios de información AIC Y BIC #
p=2
q=2
P=1
Q=1

# Entre los ordenes de diferenciacion
d=1
D=1
(maxfilas=(p+1)*(q+1)*(P+1)*(Q+1))
ic_mod=matrix(rep(NA, times=maxfilas*6), nrow=maxfilas, ncol=6)

k=1
for(i in 0:p) {
for(j in 0:q) {
for(s in 0:P) {
for(m in 0:Q) {
modelo = tryCatch(Arima(accidt, order = c(i, d, j),
                        seasonal = list(order = c(s, D, m), period = 12), method = c("CSS-ML")),
                  error = function(e) NULL)
     ic_mod[k,1]=i
     ic_mod[k,2]=j
     ic_mod[k,3]=s
     ic_mod[k,4]=m
     if (!is.null(modelo)) {
       numpar=length(coef(modelo))
       T=length(residuals(modelo))
       ic_mod[k,5]=-2*(modelo$loglik/T)+(2*numpar)/T      # AIC
       ic_mod[k,6]=-2*(modelo$loglik/T)+(numpar*log(T))/T  # BIC
     }
k=k+1
}
}
}
}
p_=ic_mod[,1]
q_=ic_mod[,2]
P_=ic_mod[,3]
Q_=ic_mod[,4]
AIC=ic_mod[,5]
BIC=ic_mod[,6]

In [ ]:
# Tabla con ordenes de los modelos ajustados y los criterios de información #
(Crit_Inf = cbind(p_, q_, P_, Q_, AIC, BIC))

In [ ]:
# Ordenamiento del criterio de información AIC de menor a mayor #
Crit_Inf=na.omit(data.frame(Crit_Inf))
(aic=head(Crit_Inf[order(Crit_Inf$AIC),], 10))

In [ ]:
# Ordenamiento del criterio de información BIC de menor a mayor #
(bic=head(Crit_Inf[order(Crit_Inf$BIC),], 10))

In [ ]:
# Mejor modelo según BIC (criterio más parsimonioso) #
ord1  = c(bic$p_[1], d, bic$q_[1])
sord1 = c(bic$P_[1], D, bic$Q_[1])
lab1  = sprintf("SARIMA(%d,%d,%d)x(%d,%d,%d)12", ord1[1], ord1[2], ord1[3], sord1[1], sord1[2], sord1[3])
cat("Modelo 1 (criterios de información):", lab1, "\n")

---
## **Conclusión**:

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

Los modelos con menor AIC y BIC aparecen en las tablas anteriores. Se selecciona como **Modelo 1** el que minimiza el **BIC**, por ser el criterio que más penaliza el número de parámetros (principio de **parsimonia**): **___** *(completar con la salida; se espera* $SARIMA(0,1,1)\times(0,1,1)_{12}$*, lo que confirmaría la identificación por ACF y PACF)*.

</div>

---

### **Usando la función auto.arima()**

In [ ]:
# Ahora vamos a utilizar la función auto.arima para calcular el modelo de forma automática #
aut = auto.arima(accidt)
aut

ao = arimaorder(aut)
ord2  = as.numeric(ao[1:3])
sord2 = if (length(ao) >= 6) as.numeric(ao[4:6]) else c(0, 0, 0)

# Si auto.arima coincide con el Modelo 1, se toma como Modelo 2 el segundo mejor por AIC #
if (all(c(ord2, sord2) == c(ord1, sord1))) {
  alt = aic[!(aic$p_==ord1[1] & aic$q_==ord1[3] & aic$P_==sord1[1] & aic$Q_==sord1[3]), ][1, ]
  ord2  = c(alt$p_, d, alt$q_)
  sord2 = c(alt$P_, D, alt$Q_)
  origen2 = "segundo mejor modelo por AIC (auto.arima coincidió con el Modelo 1)"
} else {
  origen2 = "recomendación de auto.arima()"
}
lab2 = sprintf("SARIMA(%d,%d,%d)x(%d,%d,%d)12", ord2[1], ord2[2], ord2[3], sord2[1], sord2[2], sord2[3])
cat("Modelo 2:", lab2, "->", origen2, "\n")

## **Estimación de Parámetros para Dos Modelos Tentativos**

### **Modelo 1 :** Usando los criterios de información y la componente estacional

In [ ]:
# Especificación y estimación del modelo #
(mod1 = stats::arima(accidt, order = ord1, seasonal = list(order = sord1, period = 12), method = c("CSS-ML")))

In [ ]:
# Usando la función sarima() (astsa): estimación, significancia y diagnóstico gráfico #
sar1 = sarima(accidt, p=ord1[1], d=ord1[2], q=ord1[3], P=sord1[1], D=sord1[2], Q=sord1[3], S=12)

In [ ]:
# Para mirar la significancia de las estimaciones #
sar1$ttable

### **Modelo 2 :** Usando la recomendación de auto.arima()

In [ ]:
# Especificación y estimación del modelo #
(mod2 = stats::arima(accidt, order = ord2, seasonal = list(order = sord2, period = 12), method = c("CSS-ML")))

In [ ]:
# Usando la función sarima() (astsa) #
sar2 = sarima(accidt, p=ord2[1], d=ord2[2], q=ord2[3], P=sord2[1], D=sord2[2], Q=sord2[3], S=12)

In [ ]:
# Para mirar la significancia de las estimaciones #
sar2$ttable

---
## **Observación:**

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

Un parámetro es **significativo** si su p-valor es menor que 0.05. Para el Modelo 1, los coeficientes **___** son significativos. Para el Modelo 2, los coeficientes **___** *(completar con las tablas `ttable`)*. Si algún coeficiente no es significativo, el modelo tiene parámetros que sobran y es menos parsimonioso.

</div>

---

## **Diagnóstico Residual**

In [ ]:
# Ahora nos disponemos a evaluar el ajuste de los modelos #
res_mod1 = residuals(mod1)
res_mod2 = residuals(mod2)

In [ ]:
# Valores ajustados #
ajust1 = accidt - res_mod1
ajust2 = accidt - res_mod2

In [ ]:
# Gráfico para los valores ajustados #
par(mfrow=c(2,1))
ts.plot(accidt, ajust1, col=c("black","red"), lwd=c(2,2), lty=c(1,2), main = paste("Valores ajustados - Modelo 1:", lab1)); grid()
legend("topright", legend=c("Serie", "Ajustados"), col=c("black","red"), lty=c(1,2), lwd=2, bg="white")
ts.plot(accidt, ajust2, col=c("black","blue"), lwd=c(2,2), lty=c(1,2), main = paste("Valores ajustados - Modelo 2:", lab2)); grid()
legend("topright", legend=c("Serie", "Ajustados"), col=c("black","blue"), lty=c(1,2), lwd=2, bg="white")

### **Construyendo los Criterios de Información**

In [ ]:
# Longitud de Residuales #
n1=length(residuals(mod1))
n2=length(residuals(mod2))

# Número de Parámetros #
p1=length(coef(mod1))
p2=length(coef(mod2))

# AIC #
AIC1 = -2*(mod1$loglik/n1)+(2*p1)/n1
AIC2 = -2*(mod2$loglik/n2)+(2*p2)/n2

# BIC #
BIC1 = -2*(mod1$loglik/n1)+(log(n1)*p1)/n1
BIC2 = -2*(mod2$loglik/n2)+(log(n2)*p2)/n2

# Criterios de Información para ambos modelos #
c_f = data.frame("Criterios" = c(paste("Modelo_1:", lab1), paste("Modelo_2:", lab2)),
                 "Parametros" = c(p1, p2),
                 "AIC" = c(AIC1, AIC2),
                 "BIC" = c(BIC1, BIC2))
print(c_f)

### **Análisis Residual**

In [ ]:
# ==============================================================================
# 1. EXTRACCIÓN Y RECORTES DE RESIDUOS ESTANDARIZADOS (OMITIENDO 13 PRIMEROS)
# ==============================================================================
# Se omiten las primeras d + S*D = 1 + 12 = 13 observaciones, que se pierden al diferenciar
res_est1 <- as.numeric(res_mod1[14:length(res_mod1)]) / sqrt(mod1$sigma2)
res_est2 <- as.numeric(res_mod2[14:length(res_mod2)]) / sqrt(mod2$sigma2)

# ==============================================================================
# 2. PRUEBAS ESTADÍSTICAS
# ==============================================================================
# Levene: 4 ventanas temporales para comparar la varianza en el tiempo
grupos <- cut(seq_along(res_est1), breaks = 4, labels = c("V1", "V2", "V3", "V4"))
lev_m1 <- suppressWarnings(leveneTest(res_est1 ~ grupos)$`Pr(>F)`[1])
lev_m2 <- suppressWarnings(leveneTest(res_est2 ~ grupos)$`Pr(>F)`[1])

# Dickey-Fuller sobre los residuos
adf_m1 <- suppressWarnings(tseries::adf.test(res_est1)$p.value)
adf_m2 <- suppressWarnings(tseries::adf.test(res_est2)$p.value)

# Ljung-Box hasta el rezago 24 (descontando los parámetros estimados con fitdf)
lb_m1 <- Box.test(res_est1, lag = 24, type = "Ljung-Box", fitdf = p1)$p.value
lb_m2 <- Box.test(res_est2, lag = 24, type = "Ljung-Box", fitdf = p2)$p.value

# ==============================================================================
# REPORTE FORMAL DE DECISIONES EN CONSOLA
# ==============================================================================
reporte <- function(nombre, lev, adf, lb) {
  cat("[", nombre, "]\n", sep = "")
  cat(" • Levene (Homocedasticidad): p-valor =", round(lev, 4),
      ifelse(lev > 0.05, "-> Cumple Varianza Constante (p > 0.05)\n", "-> Heterocedástico (p <= 0.05)\n"))
  cat(" • ADF (Estacionariedad)    : p-valor =", round(adf, 4),
      ifelse(adf <= 0.05, "-> Cumple Estacionariedad (p <= 0.05)\n", "-> No Estacionario (p > 0.05)\n"))
  cat(" • Ljung-Box (Independencia): p-valor =", round(lb, 4),
      ifelse(lb > 0.05, "-> Cumple Ruido Blanco / Sin Autocorrelación (p > 0.05)\n", "-> Autocorrelación Remanente (p <= 0.05)\n"))
}
cat("\n========================================================\n")
cat("          EVALUACIÓN DE RESIDUOS ESTANDARIZADOS          \n")
cat("========================================================\n")
reporte(paste("MODELO 1:", lab1), lev_m1, adf_m1, lb_m1)
cat("\n")
reporte(paste("MODELO 2:", lab2), lev_m2, adf_m2, lb_m2)
cat("========================================================\n")

In [ ]:
# Presentamos las gráficas de los residuales estandarizados #
par(mfrow=c(2,1))
plot.ts(res_est1, main=paste("Residuales Estandarizados - Modelo 1:", lab1), ylab=""); abline(h=c(-2,0,2), col=c("red","gray","red"), lty=c(2,1,2))
plot.ts(res_est2, main=paste("Residuales Estandarizados - Modelo 2:", lab2), ylab=""); abline(h=c(-2,0,2), col=c("red","gray","red"), lty=c(2,1,2))

In [ ]:
# ACF y PACF de residuales #
par(mfrow=c(2,2))
acf(res_est1, lag.max = 24, main = "ACF Residuales - Modelo 1")
pacf(res_est1, lag.max = 24, main = "PACF Residuales - Modelo 1")
acf(res_est2, lag.max = 24, main = "ACF Residuales - Modelo 2")
pacf(res_est2, lag.max = 24, main = "PACF Residuales - Modelo 2")

### **Chequeo de Normalidad**

In [ ]:
# Gráfico del Q-Q Plot con banda de confianza (car) #
par(mfrow=c(1,2))
qqPlot(res_est1, envelope = 0.95, main = paste("Q-Q Plot Modelo 1:", lab1), ylab = "Residuos estandarizados")
qqPlot(res_est2, envelope = 0.95, main = paste("Q-Q Plot Modelo 2:", lab2), ylab = "Residuos estandarizados")

In [ ]:
# ==============================================================================
# PRUEBAS DE NORMALIDAD Y REPORTE DE DECISIÓN
# ==============================================================================
# H0: Los residuos siguen una distribución normal
# H1: Los residuos NO siguen una distribución normal

# Shapiro Wilk: Prueba de Hipótesis para contrastar normalidad #
sw_m1 <- shapiro.test(res_est1)$p.value
sw_m2 <- shapiro.test(res_est2)$p.value

# Kolmogorov-Smirnov (Lilliefors): Prueba no paramétrica basada en Bondad de Ajuste #
ks_m1 <- lillie.test(res_est1)$p.value
ks_m2 <- lillie.test(res_est2)$p.value

# Jarque Bera: Basado en la Asimetría y Curtosis de una Normal #
jb_m1 <- jarqueberaTest(res_est1)@test$p.value
jb_m2 <- jarqueberaTest(res_est2)@test$p.value

tabla_norm <- data.frame(
  Prueba   = c("Shapiro-Wilk", "Kolmogorov-Smirnov (Lilliefors)", "Jarque-Bera"),
  Modelo_1 = round(as.numeric(c(sw_m1, ks_m1, jb_m1)), 4),
  Modelo_2 = round(as.numeric(c(sw_m2, ks_m2, jb_m2)), 4)
)
tabla_norm$Decision_M1 <- ifelse(tabla_norm$Modelo_1 > 0.05, "Normal", "No normal")
tabla_norm$Decision_M2 <- ifelse(tabla_norm$Modelo_2 > 0.05, "Normal", "No normal")
print(tabla_norm)

---
## **Observación:**

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

Con un nivel de significancia del 5%: si el p-valor de **Ljung-Box** es mayor que 0.05, no hay autocorrelación en los residuos y estos se comportan como **ruido blanco**. Si los p-valores de **Shapiro-Wilk, Lilliefors y Jarque-Bera** son mayores que 0.05, no se rechaza la **normalidad**, lo que respalda la validez de los intervalos de predicción. *(Completar con los resultados de cada modelo.)*

</div>

---

## **Tabla Resumen de los Modelos**

In [ ]:
# Tabla resumen: criterios de información + diagnóstico residual #
resumen <- data.frame(
  Modelo        = c(lab1, lab2),
  Parametros    = c(p1, p2),
  AIC           = round(c(AIC1, AIC2), 4),
  BIC           = round(c(BIC1, BIC2), 4),
  sigma2        = round(c(mod1$sigma2, mod2$sigma2), 4),
  LjungBox_p    = round(c(lb_m1, lb_m2), 4),
  Levene_p      = round(c(lev_m1, lev_m2), 4),
  Shapiro_p     = round(as.numeric(c(sw_m1, sw_m2)), 4)
)
print(resumen)

---
## **Selección Final del Modelo**
---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">


<h1 align="center"><b>  </b></h1>

El **Modelo ___** es la mejor opción para la serie de muertes accidentales (**USAccDeaths**) *(completar)*:

- ***Modelación de la Estacionalidad:*** la serie tiene un patrón anual claro de 12 meses, que el modelo captura con la diferencia estacional $(1-B^{12})$ y la componente estacional.
- ***Criterios de Información:*** presenta el menor AIC y BIC con el menor número de parámetros (parsimonia).
- ***Significancia:*** todos sus parámetros son significativos.
- ***Diagnóstico Residual:*** los residuos se comportan como ruido blanco (Ljung-Box), con varianza constante (Levene) y distribución aproximadamente normal.

**Ecuación del modelo** (para un $SARIMA(0,1,1)\times(0,1,1)_{12}$):

$$(1-B)(1-B^{12})\,Y_t = (1+\theta_1 B)(1+\Theta_1 B^{12})\,a_t, \qquad a_t \sim RB(0,\sigma^2)$$

con $\hat\theta_1 =$ ___ , $\hat\Theta_1 =$ ___ y $\hat\sigma^2 =$ ___ .

</div>

---

---
## **Pronóstico (Forecasting)**
---

In [ ]:
# ==============================================================================
# 1. GENERACIÓN DE PRONÓSTICOS (24 MESES: 1979 - 1980)
# ==============================================================================
h_ahead <- 24
pred1 <- predict(mod1, n.ahead = h_ahead)
pred2 <- predict(mod2, n.ahead = h_ahead)

# Llevar los pronósticos a la escala original (corrección por sesgo si se usó log) #
a_escala <- function(pred) {
  if (transf == "log") {
    data.frame(Pred  = exp(pred$pred + .5*pred$se^2),
               Lower = exp(pred$pred - 1.96*pred$se + .5*pred$se^2),
               Upper = exp(pred$pred + 1.96*pred$se + .5*pred$se^2))
  } else {
    data.frame(Pred  = pred$pred,
               Lower = pred$pred - 1.96*pred$se,
               Upper = pred$pred + 1.96*pred$se)
  }
}
pr1 <- a_escala(pred1)
pr2 <- a_escala(pred2)

tiempo_hist   <- as.numeric(time(accid))
tiempo_futuro <- as.numeric(time(pred1$pred))

df_hist <- data.frame(Tiempo = tiempo_hist, Valor = as.numeric(accid))

df_pred_total <- rbind(
  data.frame(Panel = paste("Modelo 1:", lab1), Tiempo = tiempo_futuro,
             Pred = as.numeric(pr1$Pred), Lower_95 = as.numeric(pr1$Lower), Upper_95 = as.numeric(pr1$Upper)),
  data.frame(Panel = paste("Modelo 2:", lab2), Tiempo = tiempo_futuro,
             Pred = as.numeric(pr2$Pred), Lower_95 = as.numeric(pr2$Lower), Upper_95 = as.numeric(pr2$Upper))
)
df_pred_total$Panel <- factor(df_pred_total$Panel, levels = unique(df_pred_total$Panel))

# ==============================================================================
# 2. GRÁFICO EN GGPLOT2 CON INTERVALOS DE SOMBRA (RIBBON)
# ==============================================================================
g_forecast <- ggplot() +
  geom_line(data = df_hist, aes(x = Tiempo, y = Valor, color = "Histórico"), linewidth = 0.7) +
  geom_ribbon(data = df_pred_total, aes(x = Tiempo, ymin = Lower_95, ymax = Upper_95, fill = "IC 95%"), alpha = 0.25) +
  geom_line(data = df_pred_total, aes(x = Tiempo, y = Pred, color = "Pronóstico"), linewidth = 0.9) +
  scale_color_manual(values = c("Histórico" = "#1F4E79", "Pronóstico" = "#E64A19")) +
  scale_fill_manual(values = c("IC 95%" = "#E64A19")) +
  scale_x_continuous(breaks = seq(1973, 1981, by = 1)) +
  facet_wrap(~ Panel, ncol = 1, scales = "fixed") +
  labs(
    title = "Pronóstico de Muertes Accidentales en EE. UU. a 24 Meses (1979 - 1980)",
    subtitle = "Proyecciones con Intervalos de Predicción al 95%",
    x = "Año", y = "Número de muertes", color = "Series", fill = "Tolerancia"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    plot.subtitle = element_text(size = 12, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 12),
    strip.background = element_rect(fill = "gray92", color = NA),
    legend.position = "bottom",
    panel.grid.minor = element_blank(),
    panel.grid.major = element_line(color = "gray88", linewidth = 0.4)
  )
print(g_forecast)

In [ ]:
# Listado de los pronósticos e intervalos de predicción del Modelo 1 (escala original) #
pronosticos <- data.frame(
  Fecha    = format(seq(as.Date("1979-01-01"), by = "month", length.out = h_ahead), "%b-%Y"),
  Pronostico = round(pr1$Pred, 0),
  Lim_Inf    = round(pr1$Lower, 0),
  Lim_Sup    = round(pr1$Upper, 0)
)
pronosticos

---
## **Evaluación de Pronóstico**
---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">


<h1 align="center"><b>  </b></h1>

Para validar formalmente la precisión predictiva fuera de muestra, se realiza una partición temporal de la serie dividiendo las observaciones en un 70% de datos de entrenamiento (Train) y un 30% de datos de prueba (Test). Se ajustan ambos modelos con los datos de entrenamiento, se pronostica el periodo de prueba y se comparan los pronósticos con los valores reales mediante el **MSE**, el **RMSE**, el **MAE** y el **MAPE**.

</div>

In [ ]:
# ==============================================================================
# 1. PARTICIÓN TEMPORAL DE LA SERIE (70% TRAIN / 30% TEST)
# ==============================================================================
n_total <- length(accid)
n_train <- round(n_total * 0.70)
n_test  <- n_total - n_train

accid_train <- window(accid, end = time(accid)[n_train])
accid_test  <- window(accid, start = time(accid)[n_train + 1])
if (transf == "log") { train_t <- log(accid_train) } else { train_t <- accid_train }

# ==============================================================================
# 2. RE-AJUSTE DE MODELOS EN 'TRAIN' Y PRONÓSTICO EN 'TEST'
# ==============================================================================
mod1_train <- stats::arima(train_t, order = ord1, seasonal = list(order = sord1, period = 12), method = "CSS-ML")
mod2_train <- stats::arima(train_t, order = ord2, seasonal = list(order = sord2, period = 12), method = "CSS-ML")

pt1 <- a_escala(predict(mod1_train, n.ahead = n_test))
pt2 <- a_escala(predict(mod2_train, n.ahead = n_test))

val_real <- as.numeric(accid_test)
val_p1   <- as.numeric(pt1$Pred)
val_p2   <- as.numeric(pt2$Pred)

# ==============================================================================
# 3. CÁLCULO DE MÉTRICAS DE EVALUACIÓN
# ==============================================================================
calc_mse  <- function(real, pred) mean((real - pred)^2)
calc_mae  <- function(real, pred) mean(abs(real - pred))
calc_mape <- function(real, pred) mean(abs((real - pred) / real)) * 100

metricas <- data.frame(
  Modelo = c(lab1, lab2),
  MSE    = round(c(calc_mse(val_real, val_p1),  calc_mse(val_real, val_p2)), 2),
  RMSE   = round(sqrt(c(calc_mse(val_real, val_p1), calc_mse(val_real, val_p2))), 2),
  MAE    = round(c(calc_mae(val_real, val_p1),  calc_mae(val_real, val_p2)), 2),
  MAPE   = round(c(calc_mape(val_real, val_p1), calc_mape(val_real, val_p2)), 2)
)

cat("\n========================================================\n")
cat("       EVALUACIÓN DE DESEMPEÑO FUERA DE MUESTRA         \n")
cat("   (Train: 70% -", n_train, "obs | Test: 30% -", n_test, "obs)\n")
cat("========================================================\n")
print(metricas)
cat("--------------------------------------------------------\n")
if (metricas$MAPE[1] < metricas$MAPE[2]) {
  cat("DECISIÓN FINAL: El Modelo 1 presenta mayor precisión predictiva fuera de muestra.\n")
} else {
  cat("DECISIÓN FINAL: El Modelo 2 presenta mayor precisión predictiva fuera de muestra.\n")
}
cat("========================================================\n")

In [ ]:
# ==============================================================================
# 4. VISUALIZACIÓN EN GGPLOT2 (ENTRENAMIENTO, PRUEBA Y PRONÓSTICOS)
# ==============================================================================
tiempo_total <- as.numeric(time(accid))
tiempo_train <- tiempo_total[1:n_train]
tiempo_test  <- tiempo_total[(n_train + 1):n_total]

df_real <- rbind(
  data.frame(Tiempo = tiempo_train, Valor = as.numeric(accid_train), Serie = "Entrenamiento (70%)"),
  data.frame(Tiempo = tiempo_test,  Valor = val_real,                Serie = "Test (Real 30%)")
)

df_val <- rbind(
  data.frame(Panel = paste0("Modelo 1: ", lab1, "\n(MAPE = ", metricas$MAPE[1], "%)"),
             Tiempo = tiempo_test, Pred = val_p1, Lower = as.numeric(pt1$Lower), Upper = as.numeric(pt1$Upper)),
  data.frame(Panel = paste0("Modelo 2: ", lab2, "\n(MAPE = ", metricas$MAPE[2], "%)"),
             Tiempo = tiempo_test, Pred = val_p2, Lower = as.numeric(pt2$Lower), Upper = as.numeric(pt2$Upper))
)
df_val$Panel <- factor(df_val$Panel, levels = unique(df_val$Panel))

g_val <- ggplot() +
  geom_ribbon(data = df_val, aes(x = Tiempo, ymin = Lower, ymax = Upper), fill = "#E64A19", alpha = 0.2) +
  geom_line(data = df_real, aes(x = Tiempo, y = Valor, color = Serie), linewidth = 0.8) +
  geom_line(data = df_val, aes(x = Tiempo, y = Pred, color = "Pronóstico"), linewidth = 0.9, linetype = "dashed") +
  geom_vline(xintercept = tiempo_test[1], linetype = "dotted", color = "gray40") +
  scale_color_manual(values = c("Entrenamiento (70%)" = "#1F4E79", "Test (Real 30%)" = "black", "Pronóstico" = "#E64A19")) +
  scale_x_continuous(breaks = seq(1973, 1979, by = 1)) +
  facet_wrap(~ Panel, ncol = 1) +
  labs(title = "Validación Fuera de Muestra: Pronóstico vs. Valores Reales",
       subtitle = "Partición temporal 70% entrenamiento / 30% prueba, intervalos al 95%",
       x = "Año", y = "Número de muertes", color = "") +
  theme_minimal(base_size = 14) +
  theme(
    plot.title = element_text(face = "bold", size = 16, hjust = 0.5),
    plot.subtitle = element_text(size = 12, hjust = 0.5, color = "gray30"),
    strip.text = element_text(face = "bold", size = 12),
    strip.background = element_rect(fill = "gray92", color = NA),
    legend.position = "bottom",
    panel.grid.minor = element_blank()
  )
print(g_val)

> ### **Validación de Precisión Predictiva Fuera de Muestra**
> 
> Para confirmar la capacidad de generalización y descartar sobreajuste (*overfitting*), la serie de muertes accidentales (`USAccDeaths`) fue dividida en una ventana de entrenamiento (70%, $N_{\text{train}} = 50$ observaciones, enero 1973 - febrero 1977) y una ventana de evaluación fuera de muestra (30%, $N_{\text{test}} = 22$ observaciones, marzo 1977 - diciembre 1978).
> 
> **Conclusión:** El modelo con **menor MAPE** es **___**, con un error porcentual medio de **___%** *(completar)*. Un MAPE menor al 10% indica una **precisión de pronóstico alta**.

---
## **Conclusión**
---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">


<h1 align="center"><b>  </b></h1>

Siguiendo la metodología de Box-Jenkins, la serie de muertes accidentales mensuales en EE. UU. (1973-1978) se modeló adecuadamente con un **SARIMA** ___ *(completar)*. El análisis descriptivo y la ACF mostraron una **fuerte estacionalidad anual** (picos en julio, mínimos en febrero), que exigió una diferencia ordinaria y una estacional. El modelo seleccionado minimiza los criterios de información, tiene parámetros significativos y residuos que se comportan como **ruido blanco** con distribución aproximadamente normal. Sus pronósticos reproducen el patrón estacional, y los intervalos de predicción crecen con el horizonte, como es de esperarse. En la validación fuera de muestra obtuvo un **MAPE de ___%**, lo que confirma su buena capacidad predictiva.

</div>

---
## **Aplicación Shiny**
---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

Como complemento, se desarrolló una aplicación Shiny (archivo `app.R`) que permite explorar el análisis de forma interactiva: visualizar la serie y su estacionalidad, cambiar la diferenciación y observar la ACF y la PACF, modificar los órdenes del modelo SARIMA y ver en tiempo real los parámetros, los criterios de información y el diagnóstico residual, variar el horizonte de pronóstico y evaluar la precisión fuera de muestra con distintas particiones de entrenamiento y prueba.

</div>

In [ ]:
# Ejecutar la aplicación Shiny (se abre en el navegador; para cerrarla presionar Esc o el botón Stop) #
shiny::runApp("app.R")

---
# 🎬 **¡FINAL!**
---